# 01 — Computer Vision Foundations & Bounding Box Geometry

This notebook establishes the foundational coordinate representations, geometric transformations, and core spatial metrics required for object detection pipelines.

---

## 1. Mathematical Coordinate Representations

An object detection model must localize instances by predicting spatial boundaries. Across literature and libraries, three primary bounding box conventions are used:

1. **Pascal VOC format**: Absolute pixel corner coordinates
   $$b_{\text{VOC}} = [x_{\min}, y_{\min}, x_{\max}, y_{\max}]$$
   where $0 \le x_{\min} < x_{\max} \le W$ and $0 \le y_{\min} < y_{\max} \le H$.

2. **COCO format**: Absolute top-left corner coordinates with dimensions
   $$b_{\text{COCO}} = [x_{\min}, y_{\min}, w, h]$$
   where $w = x_{\max} - x_{\min}$ and $h = y_{\max} - y_{\min}$.

3. **YOLO format**: Image-normalized center coordinates and dimensions
   $$b_{\text{YOLO}} = [x_c, y_c, w_n, h_n]$$
   where $x_c = \frac{x_{\min} + x_{\max}}{2W}$, $y_c = \frac{y_{\min} + y_{\max}}{2H}$, $w_n = \frac{w}{W}$, $h_n = \frac{h}{H}$.
   All coordinates satisfy $0 \le x_c, y_c, w_n, h_n \le 1$.

In [ ]:
import numpy as np


def voc_to_coco(bbox: list[float] | np.ndarray) -> np.ndarray:
    """Convert Pascal VOC [xmin, ymin, xmax, ymax] to COCO [xmin, ymin, w, h]."""
    xmin, ymin, xmax, ymax = bbox
    return np.array([xmin, ymin, xmax - xmin, ymax - ymin], dtype=float)


def coco_to_voc(bbox: list[float] | np.ndarray) -> np.ndarray:
    """Convert COCO [xmin, ymin, w, h] to Pascal VOC [xmin, ymin, xmax, ymax]."""
    xmin, ymin, w, h = bbox
    return np.array([xmin, ymin, xmin + w, ymin + h], dtype=float)


def voc_to_yolo(bbox: list[float] | np.ndarray, width: int, height: int) -> np.ndarray:
    """Convert Pascal VOC to normalized YOLO [xc, yc, w, h] with clamping."""
    xmin, ymin, xmax, ymax = bbox
    xc = ((xmin + xmax) / 2.0) / width
    yc = ((ymin + ymax) / 2.0) / height
    w = (xmax - xmin) / width
    h = (ymax - ymin) / height
    return np.clip(np.array([xc, yc, w, h], dtype=float), 0.0, 1.0)


def yolo_to_voc(bbox: list[float] | np.ndarray, width: int, height: int) -> np.ndarray:
    """Convert normalized YOLO [xc, yc, w, h] to absolute Pascal VOC [xmin, ymin, xmax, ymax]."""
    xc, yc, w, h = bbox
    xmin = (xc - w / 2.0) * width
    ymin = (yc - h / 2.0) * height
    xmax = (xc + w / 2.0) * width
    ymax = (yc + h / 2.0) * height
    return np.array([xmin, ymin, xmax, ymax], dtype=float)


# Verify invertible transformation
test_box = [120.0, 220.0, 240.0, 530.0]
img_w, img_h = 640, 640

yolo_rep = voc_to_yolo(test_box, img_w, img_h)
recovered_box = yolo_to_voc(yolo_rep, img_w, img_h)

assert np.allclose(test_box, recovered_box)
print(f"Original VOC:  {test_box}")
print(f"Normalized YOLO: {yolo_rep.round(4).tolist()}")
print(f"Recovered VOC: {recovered_box.round(1).tolist()}")

## 2. Vectorized Intersection over Union (IoU)

The fundamental metric of spatial overlap between two bounding boxes $A$ and $B$ is the **Jaccard Index**, or **Intersection over Union (IoU)**:

$$\text{IoU}(A, B) = \frac{\text{Area}(A \cap B)}{\text{Area}(A \cup B)} = \frac{\text{Area}(A \cap B)}{\text{Area}(A) + \text{Area}(B) - \text{Area}(A \cap B)}$$

Given an intersection rectangle bounded by coordinates:
$$x_1^{\cap} = \max(x_{1A}, x_{1B}), \quad y_1^{\cap} = \max(y_{1A}, y_{1B})$$
$$x_2^{\cap} = \min(x_{2A}, x_{2B}), \quad y_2^{\cap} = \min(y_{2A}, y_{2B})$$

The intersection area is:
$$\text{Area}(A \cap B) = \max(0, x_2^{\cap} - x_1^{\cap}) \times \max(0, y_2^{\cap} - y_1^{\cap})$$

In [ ]:
def compute_iou_single(box_a: np.ndarray, box_b: np.ndarray) -> float:
    """Compute IoU between two VOC boxes [xmin, ymin, xmax, ymax]."""
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0


def compute_iou_matrix(boxes_a: np.ndarray, boxes_b: np.ndarray) -> np.ndarray:
    """Vectorized pairwise IoU computation between N boxes and M boxes via NumPy broadcasting.

    Args:
        boxes_a: (N, 4) array in VOC format
        boxes_b: (M, 4) array in VOC format
    Returns:
        (N, M) matrix of pairwise IoU overlaps
    """
    # Expand dims for broadcasting: (N, 1, 4) and (1, M, 4)
    a = boxes_a[:, np.newaxis, :]
    b = boxes_b[np.newaxis, :, :]

    x1 = np.maximum(a[..., 0], b[..., 0])
    y1 = np.maximum(a[..., 1], b[..., 1])
    x2 = np.minimum(a[..., 2], b[..., 2])
    y2 = np.minimum(a[..., 3], b[..., 3])

    intersection = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    area_a = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1])
    area_b = (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])
    union = area_a + area_b - intersection

    return np.where(union > 0, intersection / union, 0.0)


# Test pairwise IoU
proposals = np.array(
    [
        [120, 220, 240, 530],
        [122, 218, 238, 528],
        [360, 200, 470, 520],
    ]
)
targets = np.array(
    [
        [120, 220, 240, 530],
        [360, 200, 470, 520],
    ]
)

iou_mat = compute_iou_matrix(proposals, targets)
print("Pairwise IoU Matrix (Proposals x Targets):")
print(np.round(iou_mat, 3))
assert iou_mat[0, 0] == 1.0
assert iou_mat[0, 1] == 0.0

## 3. Non-Maximum Suppression (NMS)

Object detection models typically produce hundreds of redundant candidate proposals around the same physical entity. **Non-Maximum Suppression (NMS)** eliminates redundant, highly overlapping lower-confidence detections:

```text
Algorithm Non-Maximum Suppression:
Input: Proposals B = {b_1, ..., b_n}, Scores S = {s_1, ..., s_n}, Overlap Threshold theta
Output: Filtered Keep Set D

1. Initialize D = empty
2. Sort candidate indices by confidence score descending
3. While candidates remain:
    a. Select index m with maximum confidence; add to D
    b. Compute IoU between b_m and all remaining candidates b_i
    c. Suppress any b_i where IoU(b_m, b_i) >= theta
4. Return D
```

In [ ]:
def non_max_suppression(
    boxes: np.ndarray,
    scores: np.ndarray,
    iou_threshold: float = 0.5,
    score_threshold: float = 0.25,
) -> list[int]:
    """Perform Non-Maximum Suppression on bounding box proposals.

    Args:
        boxes: (N, 4) array in VOC format [xmin, ymin, xmax, ymax]
        scores: (N,) array of detection confidence scores
        iou_threshold: IoU overlap threshold above which lower score boxes are suppressed
        score_threshold: Minimum confidence score to consider
    Returns:
        List of retained box indices sorted by score descending
    """
    # Filter by confidence threshold
    valid_mask = scores >= score_threshold
    indices = np.where(valid_mask)[0]
    if len(indices) == 0:
        return []

    # Sort indices by score descending
    order = indices[np.argsort(-scores[indices])]
    keep = []

    while len(order) > 0:
        current = order[0]
        keep.append(int(current))
        if len(order) == 1:
            break

        # Calculate IoU with remaining boxes
        current_box = boxes[current : current + 1]
        remaining_boxes = boxes[order[1:]]
        ious = compute_iou_matrix(current_box, remaining_boxes)[0]

        # Keep only boxes with IoU < threshold
        suppressed = np.where(ious < iou_threshold)[0]
        order = order[suppressed + 1]

    return keep


# Test NMS
test_boxes = np.array(
    [
        [120, 220, 240, 530],  # Main target
        [122, 218, 238, 528],  # Overlapping candidate (IoU > 0.90)
        [118, 222, 242, 532],  # Overlapping candidate (IoU > 0.85)
        [360, 200, 470, 520],  # Distinct person
        [20, 20, 80, 80],  # False positive with low score
    ]
)
test_scores = np.array([0.95, 0.88, 0.76, 0.92, 0.15])

kept = non_max_suppression(test_boxes, test_scores, iou_threshold=0.5, score_threshold=0.25)
print(f"Retained indices: {kept}")
assert kept == [
    0,
    3,
]  # Box 0 (score 0.95) and Box 3 (score 0.92) kept; Box 1, 2 suppressed; Box 4 filtered by score
print("NMS successfully suppressed redundant proposals while preserving distinct instances!")